# Summarize notebook
This notebook summerizes a beach specific database into apex lists.

Set the constants below first.

# Import helper functions

In [ ]:
%run ../0.shared_notebooks/0_helper_functions.ipynb

# Set constants

In [ ]:
#Set constants
CASE="DIVD-2025-00018"
SUB="stealers"
IN_DIR="../LEAK_DB/"
OUT_DIR="../OUT/"
IN_DB=f"{IN_DIR}/{CASE}-{SUB}.sqlite3"
MAX_UNIQUE_USERNAMES=2_000_000

In [ ]:
!echo $IN_DIR && ls -lah $IN_DIR
!echo $OUT_DIR && ls -lah $OUT_DIR 

## Open DB


In [ ]:
conn = sqlite3.connect(IN_DB)

In [ ]:
conn

## Email domains

In [ ]:
records = []
for row in conn.execute("""
    SELECT email_apex, count(username)
    FROM entity
    GROUP BY email_apex
    ORDER BY email_apex
""").fetchall() :
    records.append(row)
email_df = pd.DataFrame(records, columns = [ "email_apex", "unique_users" ])

In [ ]:
email_df.head()

In [ ]:
email_df.to_csv(f"{OUT_DIR}/{CASE}-{SUB}-email_apexs.csv", index=False)

## URL domains

In [ ]:
records = []
for row in conn.execute("""
    SELECT url_apex, count(username)
    FROM entity
    GROUP BY url_apex
    ORDER BY url_apex
""").fetchall() :
    records.append(row)
url_df = pd.DataFrame(records, columns = [ "url_apex", "unique_users" ])

In [ ]:
url_df.head()

In [ ]:
url_df.to_csv(f"{OUT_DIR}/{CASE}-{SUB}-url_apexs.csv", index=False)

# Per user

In [ ]:
emails = conn.execute("""
    SELECT count(DISTINCT username)
    FROM entity
""").fetchall()[0][0]
if emails > MAX_UNIQUE_USERNAMES :
    raise Exception(f"There are {emails:,} unique usernames in the database, this is more that the maximum number of {MAX_UNIQUE_USERNAMES:,}")
records = [[],[]]
last_email = "***start***"
record= []
count = 0
for row in conn.execute("""
    SELECT username, masked_passwd, url, extra_data
    FROM entity
    WHERE email_apex IS NOT NULL
    GROUP BY username
    ORDER BY username
""").fetchall() :
    if last_email != row[0] :
        if last_email != "***start***" :
            records[count % 2].append(record)
            count = count + 1
        last_email = row[0]
        record = [ row[0], "" ]
    record[-1] = f"{record[-1]}Username: {row[0]}\nPassword: {row[1]}\nUrl: {row[2]}\nExtra data: {row[3]}\n\n"
user_df1 = pd.DataFrame(records[0], columns=['email', 'records'])
user_df2 = pd.DataFrame(records[1], columns=['email', 'records'])


In [ ]:
user_df2.head()

In [ ]:
user_df1.to_csv(f"{OUT_DIR}/{CASE}-{SUB}-per_user_set1.csv", index=False)

In [ ]:
user_df2.to_csv(f"{OUT_DIR}/{CASE}-{SUB}-per_user_set2.csv", index=False)

In [ ]:
ls -lah $OUT_DIR 

In [ ]:
!(cd $OUT_DIR;tar -cvzf $CASE-$SUB-apexes.tgz $CASE-$SUB-*.csv)

In [ ]:
ls -lah $OUT_DIR 